# 02 — Synthetic Error Insertion on FinQA (First 5 Rows)

All config lives in [`config.py`](../config.py).  
All Groq logic lives in [`groq_utils.py`](../groq_utils.py).  
Edit those files — **never need to touch this notebook.**


In [ ]:
# ── Install dependencies if needed ────────────────────────────────────────
# %pip install -q datasets groq python-dotenv

In [ ]:
import sys, re
sys.path.insert(0, "..")   # so notebooks/ can import from FRED/

from datasets import load_dataset
import groq_utils as gu
from config import FINQA_SUBSET, RAGBENCH_REPO

# ── Init ──────────────────────────────────────────────────────────────────
API_KEY = gu.load_api_key()
MODEL   = gu.pick_model(API_KEY)
client  = gu.make_client(API_KEY)

N_ROWS  = 5

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
ds    = load_dataset(RAGBENCH_REPO, FINQA_SUBSET, trust_remote_code=True)
train = ds["train"]
print(f"FinQA train: {len(train):,} rows | columns: {train.column_names}")

In [ ]:
# ── Error insertion loop ──────────────────────────────────────────────────
results = []

for idx in range(N_ROWS):
    row = train[idx]
    documents = row.get("documents") or row.get("document") or row.get("context") or ""
    question  = row.get("question")  or row.get("query")    or ""
    response  = row.get("response")  or row.get("answer")   or row.get("output") or ""

    print(f"\n{'='*70}\nROW {idx+1} / {N_ROWS}\n{'='*70}")
    print(f"\n📄 QUESTION:\n{question}")
    print(f"\n✅ ORIGINAL RESPONSE:\n{response}")

    corrupted = gu.insert_error(client, MODEL, documents, question, response)
    print(f"\n⚠️  CORRUPTED RESPONSE (tagged):\n{corrupted}")

    results.append({"idx": idx, "question": question, "response": response, "corrupted": corrupted})

print(f"\n\n✔ Done — {len(results)} rows processed.")

In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────
TAG_PATTERN = re.compile(r"<(Temporal|Numerical|Entity|Relation|Contradictory|Unverifiable)>")

print(f"{'Row':<5} {'Tag found':<15} {'Error type'}")
print("-" * 38)
for r in results:
    match = TAG_PATTERN.search(r["corrupted"])
    etype = match.group(1) if match else "— NONE —"
    print(f"{r['idx']+1:<5} {'✅' if match else '❌':<15} {etype}")